# Bài 12 · Trực quan hoá cơ bản

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Chọn dạng biểu đồ theo câu hỏi: đường, cột, histogram hoặc phân tán.
2. Vẽ bằng `fig, ax` của matplotlib; vẽ nhanh bằng `.plot` của pandas.
3. Hoàn thiện một hình tự giải thích được: tiêu đề nêu thông điệp, trục có đơn vị, có nguồn.
4. Xuất hình từ code vào `figures/`; nhận ra cách trình bày gây hiểu sai (trục cắt, khác thang).

**Giáo trình:** [McKinney, chương 9, §9.1–9.2](https://wesmckinney.com/book/plotting-and-visualization).
Dữ liệu: Santiago, mốc chụp **29/06/2026**, hai file `visualisations/` (giá là **CLP/đêm**).
Các ví dụ được kiểm tra bằng pandas **3.0.3** và matplotlib **3.11**.

## Chuẩn bị

Ô cài đặt dành cho Colab. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.
Chạy các ô demo theo thứ tự; ô **Bài tập** để bạn tự viết lời giải và nhận xét.
Ô dưới dùng file đã tải nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn (khoảng 20 MB).

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = ("https://data.insideairbnb.com/chile/rm/santiago/"
        "2026-06-29/visualisations")
tep_ls = Path("data/listings_santiago_2026-06-29.csv")
tep_rv = Path("data/reviews_santiago_2026-06-29.csv")
df = pd.read_csv(tep_ls if tep_ls.exists() else f"{BASE}/listings.csv")
rv = pd.read_csv(tep_rv if tep_rv.exists() else f"{BASE}/reviews.csv",
                 parse_dates=["date"])

XANH, CAM = "#1E93AB", "#E8890C"     # hai màu dùng chung cho cả bài
Path("figures").mkdir(exist_ok=True)  # nơi lưu hình xuất ra
print("pandas:", pd.__version__)

## 1. Vì sao cần biểu đồ?

Bộ tứ Anscombe: bốn bộ, mỗi bộ 11 cặp (x, y). Ô dưới tính trung bình (TB), độ lệch chuẩn (ĐLC),
hệ số tương quan r và đường thẳng khớp nhất của từng bộ.

In [ ]:
ans = {
    "I": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
          [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
           [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": ([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
            [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV": ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
           [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
for ten, (x, y) in ans.items():
    x, y = np.array(x), np.array(y)
    b, a = np.polyfit(x, y, 1)            # đường thẳng khớp nhất y = a + b·x
    print(f"{ten:>3}  TB x = {x.mean():.1f}  TB y = {y.mean():.2f}  "
          f"ĐLC y = {y.std(ddof=1):.2f}  r = {np.corrcoef(x, y)[0, 1]:.2f}  "
          f"y = {a:.1f} + {b:.2f}·x")

Vẽ bốn bộ cạnh nhau, cùng đường thẳng y = 3 + 0,5·x:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, (ten, (x, y)) in zip(axes, ans.items()):
    ax.scatter(x, y, color=XANH)
    ax.plot([3, 20], [3 + 0.5 * 3, 3 + 0.5 * 20], color=CAM)   # cùng đường y = 3 + 0,5·x
    ax.set_title(f"Bộ {ten}")
plt.tight_layout()
plt.show()

Chọn dạng biểu đồ theo câu hỏi:

| Câu hỏi | Dạng biểu đồ | Ví dụ trong bài |
|---|---|---|
| Thay đổi theo thời gian thế nào? | đường (line) | số đánh giá mỗi tháng |
| Các nhóm khác nhau thế nào? | cột, thanh ngang (bar) | giá trung vị theo quận |
| Một biến số phân bố ra sao? | histogram (biểu đồ tần suất) | giá mỗi đêm |
| Hai biến số liên quan thế nào? | phân tán (scatter) | kinh độ × vĩ độ, màu theo giá |

## 2. Vẽ biểu đồ với matplotlib

`df`: mỗi hàng một chỗ ở (giá là CLP/đêm); `rv`: mỗi hàng một đánh giá (mã chỗ ở, ngày).

In [ ]:
print(df.shape, rv.shape)
print(df[["neighbourhood", "room_type", "price"]].head(3))
print(rv.head(3))

Bảng cần vẽ: số đánh giá mỗi tháng, từ 01/2023 đến 05/2026.

In [ ]:
# bỏ tháng 06/2026 chưa trọn (mốc chụp 29/06)
thang = (rv[rv["date"] < "2026-06-01"]
         .set_index("date")
         .resample("ME").size()
         .loc["2023":])
print(thang.tail(3))

`fig` là cả hình, `ax` là khung vẽ có trục. Mọi lệnh vẽ và đặt nhãn đều gọi qua `ax`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thang.index, thang.values, color="#1E93AB")
ax.set_title("Số đánh giá hằng tháng tăng nhanh từ 2023")
ax.set_ylabel("số đánh giá / tháng")
plt.show()

Bốn thành phần giúp hình tự giải thích được: tiêu đề nêu **thông điệp**, chú thích điểm cần lưu ý,
nhãn trục có **đơn vị**, **nguồn** và mốc chụp. `ax.annotate` vẽ chữ kèm mũi tên chỉ vào điểm `xy`;
`ax.text(..., transform=ax.transAxes)` đặt chữ theo toạ độ của khung (0 là trái/dưới, 1 là phải/trên).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thang.index, thang.values, color=XANH, lw=2)
ax.set_title("Số đánh giá hằng tháng tại Santiago tăng hơn bốn lần trong ba năm",
             loc="left", fontweight="bold")
ax.set_xlabel("tháng")
ax.set_ylabel("số đánh giá / tháng")

dinh = thang.idxmax()                                  # tháng có nhiều đánh giá nhất
ax.annotate(f"đỉnh {dinh:%m/%Y}: {thang.max():,}".replace(",", "."),
            xy=(dinh, thang.max()),                    # điểm được chỉ vào
            xytext=(-150, -6), textcoords="offset points",
            color=CAM, arrowprops=dict(arrowstyle="->", color=CAM))
ax.text(1, -0.2, "Nguồn: Inside Airbnb, mốc chụp 29/06/2026",
        transform=ax.transAxes, ha="right", fontsize=9, color="#777")
fig.savefig("figures/danh_gia_theo_thang.svg", bbox_inches="tight")
plt.show()

print("05/2023:", thang["2023-05"].iloc[0], " 05/2026:", thang["2026-05"].iloc[0])

Thử các tham số tuỳ biến trên toàn bộ chuỗi từ 2015 (đáy năm 2020 là thời kỳ COVID-19):

In [ ]:
thang_du = rv[rv["date"] < "2026-06-01"].set_index("date").resample("ME").size()
nam = thang_du.groupby(thang_du.index.year).sum()      # tổng theo năm, để so sánh

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(thang_du.index, thang_du.values, color="k", linestyle="-", lw=1, label="mỗi tháng")
ax.plot(thang_du.rolling(12).mean(), color=CAM, linestyle="--", lw=2,
        label="trung bình 12 tháng")
ax.set_xlim(pd.Timestamp("2015-01-01"), pd.Timestamp("2026-06-30"))
ax.set_ylabel("số đánh giá / tháng")
ax.legend()
plt.show()
print(nam.loc[2019:2025])

## 3. Biểu đồ cột, histogram và phân tán

Giá trung vị theo quận, chỉ giữ quận có từ 500 chỗ ở:

In [ ]:
tk = df.groupby("neighbourhood")["price"].agg(
    median="median", n="size")
top = tk[tk["n"] >= 500].sort_values("median")
print(top)

Thanh ngang giúp đọc tên quận dài. Chỉ một thanh tô cam để nhấn nhóm cần chú ý.

In [ ]:
mau = [CAM if q == "Lo Barnechea" else XANH for q in top.index]
nhan = [f" {v:,.0f}".replace(",", ".") for v in top["median"]]   # 426230 -> 426.230

fig, ax = plt.subplots(figsize=(8.5, 3.8))
bars = ax.barh(top.index, top["median"], color=mau, height=0.6)
ax.bar_label(bars, labels=nhan)                          # ghi giá trị ở đầu mỗi thanh
ax.set_title("Giá trung vị ở Lo Barnechea gấp 4,4 lần quận thứ hai", loc="left", fontweight="bold")
ax.set_xlabel("giá trung vị (CLP/đêm), 5 quận có ≥ 500 chỗ ở")
fig.savefig("figures/gia_theo_quan.svg", bbox_inches="tight")
plt.show()

Histogram chia trục giá thành `bins` khoảng bằng nhau rồi đếm số chỗ ở trong mỗi khoảng:

In [ ]:
gia = df["price"].dropna()

fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.hist(gia, bins=50, color=XANH)       # bins: số khoảng chia trục x
ax.set_xlabel("giá (CLP/đêm)")
ax.set_ylabel("số chỗ ở")
plt.show()
print("số chỗ ở có giá:", len(gia), " giá lớn nhất:", gia.max())

`np.log10` đổi giá sang thang log: 4 là 10.000 CLP, 5 là 100.000 CLP. Mỗi bước 1 là gấp 10 lần.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.hist(np.log10(gia), bins=50, color=XANH)
ax.set_title("Giá tập trung quanh 60.000 CLP/đêm",
             loc="left", fontweight="bold")
ax.set_xlabel("log10(giá CLP/đêm)")
ax.set_ylabel("số chỗ ở")
plt.show()
print("trung vị:", gia.median())

17.688 chỗ ở có giá, mỗi chỗ ở một điểm. Nhiều điểm chồng lên nhau (overplotting) thì dùng điểm nhỏ
và `alpha` (độ đục, từ 0 là trong suốt đến 1 là đục hẳn).

In [ ]:
diem = df.dropna(subset=["price"])
tv = diem["price"].median()
mau = np.where(diem["price"] > tv, CAM, XANH)    # từng chỗ ở: đắt hơn trung vị thì cam

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.scatter(diem["longitude"], diem["latitude"], s=6, c=mau, alpha=0.45, linewidths=0)
ax.set_aspect("equal")                           # 1 độ kinh = 1 độ vĩ trên hình
ax.set_title("Chỗ ở đắt hơn trung vị (cam) tập trung ở phía đông bắc",
             loc="left", fontweight="bold")
ax.set_xlabel("kinh độ")
ax.set_ylabel("vĩ độ")
plt.show()
print("số điểm:", len(diem), " trung vị:", tv)

`plt.subplots(1, 2, sharex=True, sharey=True)`: hai khung cạnh nhau, dùng chung thang trục x và y.

In [ ]:
co_gia = df.dropna(subset=["price"])
nc = co_gia[co_gia["room_type"] == "Entire home/apt"]
pr = co_gia[co_gia["room_type"] == "Private room"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5),
                         sharex=True, sharey=True)
axes[0].hist(np.log10(nc["price"]), bins=50, color=XANH)
axes[0].set_title(f"Nguyên căn (n = {len(nc):,})".replace(",", "."))
axes[1].hist(np.log10(pr["price"]), bins=50, color=XANH)
axes[1].set_title(f"Phòng riêng (n = {len(pr):,})".replace(",", "."))
for ax in axes:
    ax.set_xlabel("log10(giá CLP/đêm)")
axes[0].set_ylabel("số chỗ ở")
plt.tight_layout()
plt.show()

### Bài tập 1: Chọn dạng biểu đồ

Chọn dạng biểu đồ và cột dữ liệu cho mỗi câu hỏi:

1. Mỗi loại phòng chiếm bao nhiêu phần trăm số chỗ ở?
2. Đa số chỗ ở nhận ít hay nhiều đánh giá mỗi tháng (`reviews_per_month`)?
3. Chỗ ở có nhiều đánh giá trong 12 tháng qua (`number_of_reviews_ltm`) có rẻ hơn không?
4. Số đánh giá mỗi tuần trong năm 2025 thay đổi thế nào?

**Vẽ thử câu 3, không quá 6 dòng code.**

In [ ]:
# Bài tập 1, câu 3: viết lời giải ở đây

## 4. Vẽ nhanh và lưu hình

pandas vẽ trực tiếp từ Series/DataFrame: `.plot()` (đường), `.plot.barh()`, `.plot.hist()`. Kết quả trả về là `ax`.

In [ ]:
dem = df["room_type"].value_counts()
ax = dem.plot.barh()
plt.show()
print(dem)

`savefig` lưu hình; định dạng chọn theo đuôi file (`.png`, `.svg`, `.pdf`). Thư mục `figures/` đã tạo ở ô chuẩn bị.

In [ ]:
fig = ax.figure          # Figure chứa ax vừa vẽ
fig.savefig("figures/so_cho_o_theo_loai.svg",
            bbox_inches="tight")

Ảnh **raster** (`PNG`) lưu lưới điểm ảnh; ảnh **vector** (`SVG`, `PDF`) lưu đường nét và chữ.
Ô dưới lưu một hình thưa điểm và một hình dày điểm ra cả ba định dạng rồi so sánh dung lượng
(số đo đổi chút ít theo phiên bản matplotlib và font):

In [ ]:
import os

# hình thưa điểm (đường) và hình dày điểm (bản đồ 17.688 điểm)
fig_thua, ax = plt.subplots(figsize=(8.8, 3.6))
ax.plot(thang.index, thang.values, color=XANH, lw=2)
fig_day, ax = plt.subplots(figsize=(8.6, 4.6))
ax.scatter(diem["longitude"], diem["latitude"], s=6, c=mau, alpha=0.45, linewidths=0)
fig_ras, ax = plt.subplots(figsize=(8.6, 4.6))
ax.scatter(diem["longitude"], diem["latitude"], s=6, c=mau, alpha=0.45, linewidths=0,
           rasterized=True)                      # riêng lớp điểm thành pixel
plt.close("all")                                 # chỉ đo dung lượng, không hiện hình


def kich_thuoc(fig, ten):
    dong = f"{ten:9}"
    for duoi in ["png", "svg", "pdf"]:           # matplotlib chọn định dạng theo đuôi file
        tep = f"figures/{ten}.{duoi}"
        fig.savefig(tep, dpi=150)
        kb = os.path.getsize(tep) / 1024
        dong += f"  {duoi.upper()} {kb:7.0f} KB"
    print(dong)


kich_thuoc(fig_thua, "duong")
kich_thuoc(fig_day, "ban_do")
kich_thuoc(fig_ras, "ban_do_r")

`rcParams` là bộ cấu hình mặc định của matplotlib; đổi một lần thì mọi hình vẽ sau đó đều theo.

In [ ]:
plt.rcParams.update({
    "axes.spines.top": False,    # bỏ viền trên
    "axes.spines.right": False,  # bỏ viền phải
    "axes.grid": True,           # bật lưới
    "grid.alpha": 0.25,          # lưới mờ
    "font.size": 12,
})

## 5. Tránh gây hiểu sai khi trình bày

Cùng số liệu, hai cách đặt trục cho hai ấn tượng khác nhau:

In [ ]:
gia_loai = df.groupby("room_type")["price"].median() / 1000      # nghìn CLP/đêm
gia_loai = gia_loai[["Private room", "Entire home/apt", "Hotel room"]]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5))
for ax, y0 in zip(axes, [30, 0]):
    ax.bar(["Phòng riêng", "Nguyên căn", "Khách sạn"], gia_loai.values, color=XANH, width=0.55)
    ax.set_ylim(y0, 130)
    ax.set_title(f"Trục y bắt đầu từ {y0}")
    ax.set_ylabel("giá trung vị (nghìn CLP/đêm)")
plt.tight_layout()
plt.show()

ks, ph = gia_loai["Hotel room"], gia_loai["Private room"]
print(f"tỷ lệ thật {ks / ph:.1f} lần; trên trục cắt từ 30: {(ks - 30) / (ph - 30):.1f} lần")

Năm điểm kiểm tra trước khi dùng một hình:

1. **Thông điệp**: tiêu đề nói hình cho thấy điều gì.
2. **Trục**: có nhãn và đơn vị; cột bắt đầu từ 0; hình so sánh dùng chung thang.
3. **Cỡ mẫu**: mỗi nhóm có bao nhiêu quan sát (n)?
4. **Dạng và màu**: dạng hợp câu hỏi; màu chỉ dùng khi mang thông tin.
5. **Nguồn và diễn giải**: ghi nguồn, mốc chụp; lời diễn giải không vượt quá dữ liệu.

### Bài tập 2: Hoàn thiện hình nháp

Hoàn thiện hình nháp ở mục 4 (`dem.plot.barh()`) bằng `fig, ax`, theo năm điểm kiểm tra:

- trục x là **% số chỗ ở**, ghi n;
- tiêu đề nêu thông điệp;
- ghi nguồn và mốc chụp;
- lưu vào `figures/`.

Xong đổi bài với bạn bên cạnh để kiểm tra năm điểm.

In [ ]:
# Bài tập 2: viết lời giải ở đây
ty_le = None   # % số chỗ ở theo loại phòng

### Đọc thêm

1. [McKinney, chương 9, §9.1–9.2](https://wesmckinney.com/book/plotting-and-visualization)
2. Berkeley Data 100: [Visualization I](https://ds100.org/course-notes/visualization-1), [Visualization II](https://ds100.org/course-notes/visualization-2)
3. [VanderPlas, Python Data Science Handbook, phần 4](https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html)
4. [matplotlib: Anatomy of a figure](https://matplotlib.org/stable/gallery/showcase/anatomy.html)

---

## Bài tập thêm (tự làm)

### Bài tập 3: Hai mốc chụp trên cùng thang đo

Tải bản `visualisations/listings.csv` của mốc chụp **27/09/2025**. Vẽ hai histogram giá (thang log10)
cho hai mốc chụp cạnh nhau, **chung thang trục** (`sharex=True, sharey=True`).
Phân phối dịch chuyển thế nào? Trước khi kết luận "giá tăng", hãy kiểm tra: các chỗ ở có mặt ở cả hai mốc
chụp có tăng giá giống nhau không? Mốc 09/2025 và 06/2026 rơi vào mùa nào ở Santiago?

In [ ]:
tep_t9 = Path("data/listings_santiago_2025-09-27.csv")
URL_T9 = ("https://data.insideairbnb.com/chile/rm/santiago/"
          "2025-09-27/visualisations/listings.csv")
t9 = pd.read_csv(tep_t9 if tep_t9.exists() else URL_T9)

# Bài tập 3: viết lời giải ở đây


### Bài tập 4: Thử số khoảng `bins`

Vẽ histogram `np.log10(gia)` với `bins=5`, `bins=50` và `bins=500` trên ba khung cạnh nhau.
Khung nào làm mất hình dạng, khung nào nhiễu?

In [ ]:
# Bài tập 4: viết lời giải ở đây


### Luyện tập thêm với thành phố của nhóm

Với thành phố của nhóm bạn, vẽ bốn hình: số đánh giá mỗi tháng, xếp hạng quận theo một chỉ số,
phân phối giá trên thang log và bản đồ điểm theo toạ độ. Mỗi hình qua năm điểm kiểm tra, lưu vào
`figures/`, kèm hai câu diễn giải ngay dưới hình.

---

## Tóm tắt bài học

- Thống kê tóm tắt che mất hình dạng; chọn dạng biểu đồ theo câu hỏi.
- `fig, ax`: đường, thanh ngang, histogram, phân tán; nhiều khung thì dùng chung thang.
- Giá dồn cục thì vẽ trên thang log; nhiều điểm thì điểm nhỏ và `alpha`.
- `.plot` để khám phá; `savefig` vào `figures/` để hình sinh lại được bằng code.
- Năm điểm kiểm tra: thông điệp, trục, cỡ mẫu, dạng và màu, nguồn và diễn giải.

Hoàn thành hai bài tập rồi chạy **Restart session and run all**.

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

**Bài sau:** seaborn để so sánh phân phối giữa các nhóm, bản đồ theo quận và phản biện biểu đồ.